# Parda — real-world test: unseen layouts + phone photos

The Phase 5 benchmark used the same 5 document templates the models were trained on. Here the models see
**4 document types they never saw** (bank statement, KYC form, pharmacy bill, offer letter) that were
**photographed with a phone** (desk, tilt, shadow, blur, JPEG). Same end-to-end pixel scoring as Phase 5.

**Settings:** Accelerator **GPU T4** (x1 or x2) · Internet **On**
**Inputs:** **Datasets tab →** `celeba-dataset` (jessicali9530) — held-out faces for the ID photos.
*(The Phase 1 notebook is not needed here.)* Optional: your own phone photos, see cell 10.
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run cells 1–4 by hand, then **Save Version → Save & Run All** (~45 min).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts, Tesseract (+Hindi, Kannada), EasyOCR, GLiNER, YOLO, PyMuPDF, iPhone .heic support
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh
!pip install -q gliner ultralytics pymupdf pillow-heif
!python -c "import gliner, ultralytics, pymupdf, torch; print('gliner', gliner.__version__, '| ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.get_device_name(0))"


In [ ]:
# 3. Held-out face photos + your models on Hugging Face
from huggingface_hub import HfApi
from parda.vision.faces import FacePool, find_faces_dir
FACES = find_faces_dir("/kaggle/input")
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
GLINER_FT, YOLO_REPO = f"{HF_USER}/parda-gliner-v1", f"{HF_USER}/parda-yolo-v1"
GLINER_BASE = "urchade/gliner_multi_pii-v1"
print("FACES =", FACES, "|", FacePool(FACES).sizes() if FACES else "none")
print("models:", GLINER_FT, "|", YOLO_REPO)
assert FACES, "Add Input -> Datasets tab -> 'celeba-dataset' (jessicali9530)"
W = "/kaggle/working"


In [ ]:
# 4. Tests (~1 min): every line ok
!python tests/test_ood.py


In [ ]:
# 5. Generate 300 unseen-layout pages (100 per language, 75 per new document type), photographed (~2 min)
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 | head -40
import json, glob
from IPython.display import Image as IPImage, Markdown, display
recs = [json.loads(l) for l in open(f"{W}/ood/annotations.jsonl")]
for t in ["bank_statement", "kyc_form", "pharmacy_bill", "offer_letter"]:
    r = next(r for r in recs if r["doc_type"] == t)
    print(t, r["lang"]); display(IPImage(f"{W}/ood/{r['image']}", width=420))


In [ ]:
# 6. END-TO-END on the unseen pages (~35 min; OCR is cached, re-running continues)
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {GLINER_FT} --gliner_base {GLINER_BASE} --yolo {YOLO_REPO} --cache {W}/ood_cache --out {W}/ood_eval --title "Unseen layouts + phone photos" 2>&1 | grep -v Warning | tail -40
display(Markdown(open(f"{W}/ood_eval/report_e2e.md").read()))


In [ ]:
# 7. Seen templates (Phase 5) vs unseen layouts + phone photos, side by side
ood = json.load(open(f"{W}/ood_eval/report_e2e.json"))
seen_path = "results/phase5/report_e2e.json"
seen = {r["config"]: r for r in json.load(open(seen_path))["configs"]} if os.path.exists(seen_path) else {}
pct = lambda v: "–" if v is None else f"{100 * v:.1f}%"
rows = ["| Configuration | All PII: seen → unseen | Pages with no leak: seen → unseen | Over-redaction: seen → unseen |",
        "|---|---|---|---|"]
for r in ood["configs"]:
    s = seen.get(r["config"], {})
    rows.append(f"| {r['config']} | {pct(s.get('fully_redacted'))} → **{pct(r['fully_redacted'])}** | "
                f"{pct(s.get('pages_clean'))} → **{pct(r['pages_clean'])}** | {pct(s.get('over_redaction'))} → {pct(r['over_redaction'])} |")
comparison = "\n".join(rows)
open(f"{W}/ood_eval/seen_vs_unseen.md", "w").write(comparison + "\n")
display(Markdown(comparison if seen else "(results/phase5/report_e2e.json not in the repo: showing unseen only)\n\n" + comparison))


In [ ]:
# 8. DEMO: one page of each new type, before | after (full pipeline)
from PIL import Image
from parda.pipeline.redact import build_redactor, redact_file
red = build_redactor("tesseract,easyocr", GLINER_FT, 0.3, True, YOLO_REPO)
os.makedirs(f"{W}/ood_demo", exist_ok=True)
for t in ["bank_statement", "kyc_form", "pharmacy_bill", "offer_letter"]:
    r = next(r for r in recs if r["doc_type"] == t)
    src = f"{W}/ood/{r['image']}"
    pdf, counts = redact_file(src, red, f"{W}/ood_demo", lang=r["lang"], png=True)
    a = Image.open(src).convert("RGB")
    b = Image.open(f"{W}/ood_demo/{os.path.splitext(os.path.basename(src))[0]}_p1_redacted.png")
    pair = Image.new("RGB", (a.width + b.width + 20, max(a.height, b.height)), "white")
    pair.paste(a, (0, 0)); pair.paste(b, (a.width + 20, 0))
    pair.save(f"{W}/ood_demo/{t}_before_after.jpg", quality=85)
    print(t, r["lang"], dict(counts)); display(IPImage(f"{W}/ood_demo/{t}_before_after.jpg", width=900))


In [ ]:
# 9. PRINT PACK (optional real-camera test): 8 clean unseen pages as one PDF. Download print_pack.pdf from
#    the Output tab, print it, photograph the pages with your phone, then see cell 10.
!python -m parda.synth.ood --n 8 --out {W}/print_pages --faces_dir {FACES} --workers 2 --clean --seed 99 > /dev/null
pages = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"{W}/print_pages/images/*.jpg"))]
pages[0].save(f"{W}/print_pack.pdf", save_all=True, append_images=pages[1:], resolution=100)
print(len(pages), "pages ->", f"{W}/print_pack.pdf")


In [ ]:
# 10. YOUR PHONE PHOTOS (optional): upload them as a dataset whose name contains "phone-photos"
#     (Add Input -> Upload), then run this cell. JPG, PNG and iPhone HEIC all work. Otherwise it just skips.
from parda.vision.faces import IMG_EXT
photo_dirs = [d for d in glob.glob("/kaggle/input/**/", recursive=True) if "phone-photos" in d.lower() or "phone_photos" in d.lower()]
photos = sorted(p for d in photo_dirs for p in glob.glob(d + "*") if p.lower().endswith(IMG_EXT + (".heic", ".heif")))
if not photos:
    print("no phone photos attached - skipping (optional step)")
else:
    os.makedirs(f"{W}/my_photos", exist_ok=True)
    for p in photos[:12]:
        pdf, counts = redact_file(p, red, f"{W}/my_photos", lang="auto", png=True)
        name = os.path.splitext(os.path.basename(p))[0]
        print(os.path.basename(p), dict(counts)); display(IPImage(f"{W}/my_photos/{name}_p1_redacted.png", width=500))


In [ ]:
# 11. Collect results to commit to GitHub (download the folder from the Output tab)
R = f"{W}/ood_results"
!mkdir -p {R}
!cp {W}/ood_eval/report_e2e.md {W}/ood_eval/report_e2e.json {W}/ood_eval/seen_vs_unseen.md {W}/ood/stats.json {R}/
!cp {W}/ood_demo/*_before_after.jpg {R}/
!ls -la {R}
